In [2]:
import pandas as pd

df = pd.read_json("../datas/vps/tanner_report.json", lines=True)

df.head(5)

,method,path,headers,uuid,peer,status,cookies,response_msg,timestamp,post_data
0,GET,/,"{'host': 'localhost', 'user-agent': 'curl/7.68...",64644ea7-b358-4208-9b28-e5aaad5e00de,"{'ip': '172.18.0.1', 'port': 53212}",200,{'sess_uuid': None},"{'version': '0.6.0', 'response': {'message': {...",2025-06-14 07:30:36.539037,NaN
1,GET,/,"{'host': '202.10.35.215', 'connection': 'keep-...",64644ea7-b358-4208-9b28-e5aaad5e00de,"{'ip': '180.248.32.210', 'port': 30702}",200,{'sess_uuid': '7ee68b0c-9f55-47e5-b872-078d30b...,"{'version': '0.6.0', 'response': {'message': {...",2025-06-14 07:31:01.702421,NaN
2,GET,/stylesheets/jquery/jquery-ui-1.11.0.css?15286...,"{'host': '202.10.35.215', 'connection': 'keep-...",64644ea7-b358-4208-9b28-e5aaad5e00de,"{'ip': '180.248.32.210', 'port': 30702}",200,{'sess_uuid': 'c3f38f1d-a1be-477a-8692-9369ce7...,"{'version': '0.6.0', 'response': {'message': {...",2025-06-14 07:31:01.860896,NaN
3,GET,/stylesheets/application.css?1528612569,"{'host': '202.10.35.215', 'connection': 'keep-...",64644ea7-b358-4208-9b28-e5aaad5e00de,"{'ip': '180.248.32.210', 'port': 30709}",200,{'sess_uuid': 'c3f38f1d-a1be-477a-8692-9369ce7...,"{'version': '0.6.0', 'response': {'message': {...",2025-06-14 07:31:02.008505,NaN
4,GET,/stylesheets/responsive.css?1528612569,"{'host': '202.10.35.215', 'connection': 'keep-...",64644ea7-b358-4208-9b28-e5aaad5e00de,"{'ip': '180.248.32.210', 'port': 8026}",200,{'sess_uuid': 'c3f38f1d-a1be-477a-8692-9369ce7...,"{'version': '0.6.0', 'response': {'message': {...",2025-06-14 07:31:02.020142,NaN


In [4]:
methods_matrix = []
tmp = []
sess_uuids_methods = {
  # "{sess_uuid}": []
}
sess_uuids = []

for _, row in df.iterrows():
  sess_uuid = row.get("cookies", {}).get("sess_uuid") or row.get("response_msg", {}).get("response", {}).get("message", {}).get("sess_uuid")
  method = row["method"]

  if sess_uuid in sess_uuids:
    sess_uuids_methods[sess_uuid].append(method)
  else:
    sess_uuids_methods[sess_uuid] = [method]
    sess_uuids.append(sess_uuid)

methods_matrix = list(sess_uuids_methods.values())
# for paths in sess_uuids_methods.values():
#   methods_matrix.append(paths)

sess_uuids_methods
# methods_matrix
# print(len(methods_matrix), len(sess_uuids_methods.keys()))

{'806c8d58-b04f-4634-b107-34877af9d1d0': ['GET'],
 '7ee68b0c-9f55-47e5-b872-078d30b9116b': ['GET'],
 'c3f38f1d-a1be-477a-8692-9369ce7d2c44': ['GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET'],
 'f278a7c1-0785-48d7-bf05-539b1f48cc0c': ['GET'],
 '35356d10-0d1c-47ab-b018-36f62dd2a08f': ['GET'],
 '1e975ea3-2f2c-4183-a4ca-7637ddbe1eec': ['GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET'],
 '7c28e45f-7aaa-42fa-b44d-0216a25150ab': ['GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET'],
 '47685c7c-17cb-4303-8a1e-5e99a9e1f9bd': ['GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET'],
 '09e54ba6-8322-4a86-afa8-a01dc89cb59b': ['GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET'],
 '98ea628e-45bf-4a62-a0ee-3fab57dbf91d': ['GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET',
  'GET'],
 'fd70efc6-4bea-4488-9b48-259dedb17253': ['POST'],
 '832ea6af-7ac1-4971-b205-2028efaef538': ['GET'],
 'a

In [5]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpmax, fpgrowth

te = TransactionEncoder()
te_ary = te.fit(methods_matrix).transform(methods_matrix)
df = pd.DataFrame(te_ary, columns=te.columns_)

df

,GET,POST
0,True,False
1,True,False
2,True,False
3,True,False
4,True,False
...,...,...
97,True,False
98,True,False
99,True,False
100,True,False


In [6]:
# frequent_itemsets = fpgrowth(df, min_support=0.3, use_colnames=True)
frequent_itemsets = fpgrowth(df, min_support=0.00001, use_colnames=True)
### alternatively:
#frequent_itemsets = apriori(df, min_support=0.6, use_colnames=True)
#frequent_itemsets = fpmax(df, min_support=0.6, use_colnames=True)

frequent_itemsets.head(20)

,support,itemsets
0,0.970588,(GET)
1,0.049020,(POST)
2,0.019608,"(GET, POST)"


In [7]:
import psycopg2

conn = psycopg2.connect(database="web_honeypot_vps", user = "postgres", password = "admin", host = "127.0.0.1", port = "5432")

print("Opened database successfully")

Opened database successfully


In [8]:
# create table
cur = conn.cursor()
cur.execute('''CREATE TABLE assoc_rules_methods (
            ID INT PRIMARY KEY     NOT NULL,
            SUPPORT           REAL    NOT NULL,
            METHOD            VARCHAR(255)     NOT NULL);''')

print("Table created successfully")

conn.commit()

Table created successfully


In [9]:
# insert data

cur = conn.cursor()

insert_query = """
    INSERT INTO assoc_rules_methods (ID, SUPPORT, METHOD)
    VALUES (%s, %s, %s)
"""

for idx, row in frequent_itemsets.iterrows():
    cur.execute(
        insert_query,
        (int(idx), float(row["support"]), str(list(row["itemsets"])))
    )

conn.commit()
print("Records created successfully")
conn.close()

Records created successfully
